[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/065.%20The%203D%20Pallet_Case%20Packing%20Problem/P65-Tier-9_executed.ipynb)

---


# 65. The 3D Pallet/Case Packing Problem

## Tier 9: A QUBO Formulation and a Classical Demonstration

### Purpose and scope

This notebook formulates a discretized, single-bin, three-dimensional packing problem as a quadratic unconstrained binary optimization (QUBO) model. It then demonstrates the model with a classical random-sampling surrogate and a classical postprocessing heuristic. The sampler is **not** a quantum annealer, and the results do not measure quantum hardware performance or establish quantum advantage. The QUBO formulation is useful for studying how packing decisions and penalties can be represented with binary variables.

The input is a list of fixed-orientation item dimensions, one bin's dimensions, and a grid resolution. The output is a set of item positions, a QUBO matrix, diagnostic plots, and measures of assignment and volume utilization.

### Modeling assumptions

- All dimensions use the same linear unit; the example uses centimeters. Volumes therefore use cubic centimeters.
- An item may be placed only at a grid point and keeps the orientation given by its input dimensions. Rotation is not modeled.
- The bin is represented by one rectangular coordinate system. Items may touch at faces, edges, or corners; positive-volume overlap is forbidden.
- The model does not impose load-bearing, stability, weight, fragility, accessibility, or unloading constraints.
- Boundary and overlap conditions appear as QUBO penalties. The postprocessing stage separately checks geometric feasibility.
- This is a one-bin placement model. It does not minimize the number of bins.

### Method overview

1. Discretize possible lower-corner positions on a regular grid.
2. Create one binary variable for every item-position pair.
3. Construct a QUBO matrix containing assignment, overlap, boundary, and height terms.
4. Sample binary vectors with a classical random sampler to illustrate energy evaluation. No quantum hardware or annealing schedule is used.
5. Convert the sampled vector into placements and use the included classical postprocessing and local search to obtain a geometrically feasible arrangement where possible.
6. Inspect the packing, the QUBO matrix, and the distinction between model size and actual hardware performance.

### QUBO model

Let \(I=\{0,\ldots,n-1\}\) be the items and \(P=\{0,\ldots,m-1\}\) the discretized candidate positions in the bin. The binary variable \(x_{i,p}\) is 1 when item \(i\) is assigned to position \(p\), and 0 otherwise. A position identifies the item's lower corner; the item dimensions determine its occupied rectangular region.

The model's energy is

$$
E(x)=P_{\mathrm{assignment}}\sum_{i\in I}\left(\sum_{p\in P}x_{i,p}-1\right)^2
+P_{\mathrm{overlap}}\sum_{(i,p)<(j,q)\in C}x_{i,p}x_{j,q}
+P_{\mathrm{boundary}}\sum_{(i,p)\in B}x_{i,p}
+P_{\mathrm{objective}}\sum_{i\in I}\sum_{p\in P}z_p x_{i,p}.
$$

Here, \(C\) is the set of pairs of candidate placements for different items whose occupied rectangular regions overlap; \(B\) is the set of item-position choices that cross a bin boundary; and \(z_p\) is the grid height of position \(p\). The first term encourages exactly one position per item, the second discourages simultaneous overlapping placements, the third discourages placements outside the bin, and the last prefers lower positions. Penalty weights determine the relative costs; a finite penalty does not, by itself, guarantee that every minimum-energy vector is feasible.

The QUBO matrix \(Q\) is symmetric and uses the convention \(E(x)=x^TQx\). For binary variables, \(x^2=x\), so the assignment term contributes a diagonal coefficient \(-P_{\mathrm{assignment}}\) for each item-position variable and a coefficient \(P_{\mathrm{assignment}}\) to each of the two symmetric off-diagonal entries for a pair of positions of the same item. The omitted constant \(nP_{\mathrm{assignment}}\) does not affect which vectors minimize energy. An overlap pair receives \(P_{\mathrm{overlap}}/2\) in each symmetric matrix entry, so its total coefficient in \(x^TQx\) is \(P_{\mathrm{overlap}}\).

The matrix has \(nm\) binary variables and \((nm)^2\) entries when stored densely. This is a model-size count, not a qubit-capacity or runtime guarantee. Even when the matrix is sparse mathematically, dense storage requires quadratic memory in the number of binary variables.

### Interpreting the demonstration

Random binary sampling is included to make the QUBO energy calculation concrete; it is not a search method that should be mistaken for quantum annealing. Postprocessing repairs sampled assignments using geometric checks and a height-based local search. The classical greedy result is included as a simple reference placement, not as a benchmark for the best classical algorithm. Measured runtimes apply only to the code executed in this notebook.

In [ ]:
import time
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# A fixed seed makes the random-sampling demonstration reproducible.
random_seed = 42
np.random.seed(random_seed)

In [ ]:
class QuantumQUBOPacking:
    """Build a QUBO packing model and demonstrate it with classical sampling."""

    def __init__(self, items, bin_dims, grid_resolution=1.0):
        if not items or len(bin_dims) != 3:
            raise ValueError("Provide at least one item and three bin dimensions.")
        if grid_resolution <= 0 or any(d <= 0 for d in bin_dims):
            raise ValueError("Bin dimensions and grid resolution must be positive.")
        if any(len(item) != 3 or any(d <= 0 for d in item) for item in items):
            raise ValueError("Each item must have three positive dimensions.")

        self.items = [tuple(map(float, item)) for item in items]
        self.bin_dims = tuple(map(float, bin_dims))
        self.L, self.W, self.H = self.bin_dims
        self.n_items = len(self.items)
        self.grid_resolution = float(grid_resolution)
        self.grid_L = int(np.ceil(self.L / self.grid_resolution))
        self.grid_W = int(np.ceil(self.W / self.grid_resolution))
        self.grid_H = int(np.ceil(self.H / self.grid_resolution))
        self.n_positions = self.grid_L * self.grid_W * self.grid_H

        self.P_assignment = 1000.0
        self.P_overlap = 500.0
        self.P_boundary = 200.0
        self.P_objective = 1.0

        # These are logical binary variables, not a claim about physical qubits.
        self.n_qubits_required = self.n_items * self.n_positions
        self.Q_matrix = None
        self.variable_mapping = {}
        self.quantum_solution = None
        self.classical_solution = None
        self.performance_metrics = {}

        print("QUBO packing model initialized:")
        print(f"  Items: {self.n_items}")
        print(f"  Bin dimensions: {self.bin_dims}")
        print(f"  Grid resolution: {self.grid_resolution:g} length units")
        print(f"  Candidate positions: {self.n_positions}")
        print(f"  Logical binary variables: {self.n_qubits_required}")

    def create_variable_mapping(self):
        """Map each (item, position) pair to its binary-vector index."""
        self.variable_mapping = {
            (item_idx, pos_idx): item_idx * self.n_positions + pos_idx
            for item_idx in range(self.n_items)
            for pos_idx in range(self.n_positions)
        }
        return self.variable_mapping

    def position_to_coordinates(self, pos_idx):
        """Return integer grid coordinates, with x varying fastest."""
        x = pos_idx % self.grid_L
        y = (pos_idx // self.grid_L) % self.grid_W
        z = pos_idx // (self.grid_L * self.grid_W)
        return x, y, z

    def coordinates_to_position(self, x, y, z):
        """Convert integer grid coordinates to a candidate-position index."""
        return x + y * self.grid_L + z * self.grid_L * self.grid_W

    def _placement_box(self, item_idx, pos_idx):
        """Return the item's lower and upper corners in physical coordinates."""
        x, y, z = self.position_to_coordinates(pos_idx)
        x0, y0, z0 = (v * self.grid_resolution for v in (x, y, z))
        length, width, height = self.items[item_idx]
        return (x0, y0, z0), (x0 + length, y0 + width, z0 + height)

    def check_overlap(self, item1_idx, pos1_idx, item2_idx, pos2_idx):
        """Check intersection of two axis-aligned item boxes."""
        lower1, upper1 = self._placement_box(item1_idx, pos1_idx)
        lower2, upper2 = self._placement_box(item2_idx, pos2_idx)
        return all(
            lower1[axis] < upper2[axis] and lower2[axis] < upper1[axis]
            for axis in range(3)
        )

    def check_boundary_violation(self, item_idx, pos_idx):
        """Return whether the item extends beyond any bin boundary."""
        _, upper = self._placement_box(item_idx, pos_idx)
        return any(upper[axis] > self.bin_dims[axis] for axis in range(3))

    def build_qubo_matrix(self):
        """Build symmetric Q for the energy convention E(x) = x.T @ Q @ x."""
        start_time = time.time()
        self.create_variable_mapping()
        n_vars = len(self.variable_mapping)
        Q = np.zeros((n_vars, n_vars), dtype=float)

        # The assignment square has a constant term that does not affect minimizers.
        for item_idx in range(self.n_items):
            indices = [
                self.variable_mapping[(item_idx, pos_idx)]
                for pos_idx in range(self.n_positions)
            ]
            Q[indices, indices] -= self.P_assignment
            for first, second in combinations(indices, 2):
                Q[first, second] += self.P_assignment
                Q[second, first] += self.P_assignment

        # Each conflict pair contributes one P_overlap coefficient to x.T @ Q @ x.
        for item1_idx in range(self.n_items):
            for item2_idx in range(item1_idx + 1, self.n_items):
                for pos1_idx in range(self.n_positions):
                    for pos2_idx in range(self.n_positions):
                        if self.check_overlap(
                            item1_idx, pos1_idx, item2_idx, pos2_idx
                        ):
                            first = self.variable_mapping[(item1_idx, pos1_idx)]
                            second = self.variable_mapping[(item2_idx, pos2_idx)]
                            Q[first, second] += self.P_overlap / 2
                            Q[second, first] += self.P_overlap / 2

        for item_idx in range(self.n_items):
            for pos_idx in range(self.n_positions):
                var_idx = self.variable_mapping[(item_idx, pos_idx)]
                x, y, z = self.position_to_coordinates(pos_idx)
                if self.check_boundary_violation(item_idx, pos_idx):
                    Q[var_idx, var_idx] += self.P_boundary
                Q[var_idx, var_idx] += self.P_objective * z

        self.Q_matrix = Q
        nonzero_count = np.count_nonzero(Q)
        print(f"QUBO matrix built in {time.time() - start_time:.2f} seconds")
        print(f"  Shape: {Q.shape}")
        print(f"  Non-zero entries: {nonzero_count:,}")
        print(f"  Density: {nonzero_count / Q.size:.2%}")
        return Q

    def calculate_qubo_energy(self, solution, Q):
        """Evaluate the QUBO energy using the symmetric-matrix convention."""
        vector = np.asarray(solution, dtype=float)
        if vector.shape != (Q.shape[0],):
            raise ValueError("The binary vector length must match the QUBO size.")
        return float(vector @ Q @ vector)

    def simulate_quantum_annealing(self, Q, num_reads=1000):
        """Use classical random binary sampling as a transparent QUBO demonstration."""
        if num_reads <= 0:
            raise ValueError("num_reads must be positive.")
        print("\nClassical random-sampling demonstration (not quantum annealing):")
        print(f"  Binary variables: {Q.shape[0]}")
        print(f"  Random samples: {num_reads}")

        solutions = np.random.randint(0, 2, size=(num_reads, Q.shape[0]))
        energies = np.einsum("bi,ij,bj->b", solutions, Q, solutions, optimize=True)
        best_idx = int(np.argmin(energies))
        print(f"  Lowest sampled energy: {energies[best_idx]:.2f}")
        print(f"  Mean sampled energy: {np.mean(energies):.2f}")
        return solutions[best_idx], float(energies[best_idx]), solutions, energies

    def get_assigned_position(self, item_idx, solution):
        """Return the first active position for an item, if one exists."""
        for pos_idx in range(self.n_positions):
            var_idx = self.variable_mapping[(item_idx, pos_idx)]
            if solution[var_idx] == 1:
                return pos_idx
        return None

    def _position_is_feasible(self, item_idx, pos_idx, assignments):
        if self.check_boundary_violation(item_idx, pos_idx):
            return False
        return not any(
            self.check_overlap(item_idx, pos_idx, other_idx, other_pos)
            for other_idx, other_pos in assignments.items()
            if other_idx != item_idx
        )

    def find_best_feasible_position(self, item_idx, current_solution=None,
                                    assignments=None):
        """Find a non-overlapping in-bin position with the lowest grid height."""
        assignments = assignments or {}
        feasible_positions = [
            pos_idx for pos_idx in range(self.n_positions)
            if self._position_is_feasible(item_idx, pos_idx, assignments)
        ]
        if not feasible_positions:
            return None
        return min(
            feasible_positions,
            key=lambda pos_idx: self.position_to_coordinates(pos_idx)[2],
        )

    def local_search_improvement(self, assignments):
        """Move assigned items to lower feasible positions when possible."""
        improved = True
        iterations = 0
        max_iterations = 100
        while improved and iterations < max_iterations:
            improved = False
            iterations += 1
            for item_idx in sorted(assignments):
                current_pos = assignments[item_idx]
                current_height = self.position_to_coordinates(current_pos)[2]
                candidate = self.find_best_feasible_position(
                    item_idx, assignments=assignments
                )
                if candidate is not None:
                    candidate_height = self.position_to_coordinates(candidate)[2]
                    if candidate_height < current_height:
                        assignments[item_idx] = candidate
                        improved = True
        print(f"  Local search completed in {iterations} iteration(s).")
        return assignments

    def classical_postprocessing(self, sampled_solution):
        """Decode samples, repair placements, then apply height-based local search."""
        assignments = {}
        for item_idx in range(self.n_items):
            sampled_pos = self.get_assigned_position(item_idx, sampled_solution)
            if (sampled_pos is not None
                    and self._position_is_feasible(item_idx, sampled_pos, assignments)):
                assignments[item_idx] = sampled_pos
            else:
                feasible_pos = self.find_best_feasible_position(
                    item_idx, assignments=assignments
                )
                if feasible_pos is not None:
                    assignments[item_idx] = feasible_pos
        return self.local_search_improvement(assignments)

    def identify_strongly_connected_components(self, Q):
        """Find connected components in the undirected nonzero QUBO graph."""
        adjacency = Q != 0
        adjacency = adjacency | adjacency.T
        visited = np.zeros(Q.shape[0], dtype=bool)
        components = []
        for start in range(Q.shape[0]):
            if visited[start]:
                continue
            stack = [start]
            visited[start] = True
            component = []
            while stack:
                node = stack.pop()
                component.append(node)
                for neighbor in np.flatnonzero(adjacency[node] & ~visited):
                    visited[neighbor] = True
                    stack.append(int(neighbor))
            components.append(component)
        return components

    def evaluate_solution_quality(self, assignments):
        """Combine assignment coverage and normalized mean grid height."""
        if not assignments:
            return 0.0
        assignment_score = len(assignments) / self.n_items
        mean_height = np.mean([
            self.position_to_coordinates(pos_idx)[2]
            for pos_idx in assignments.values()
        ])
        height_score = 1 - mean_height / max(self.grid_H, 1)
        return float(0.7 * assignment_score + 0.3 * height_score)

    def solve_quantum_packing(self, num_reads=1000):
        """Run QUBO construction, classical sampling, and classical postprocessing."""
        print("=" * 60)
        print("QUBO 3D PACKING DEMONSTRATION")
        print("=" * 60)
        Q = self.build_qubo_matrix()
        components = self.identify_strongly_connected_components(Q)
        print(f"Nonzero-graph connected components: {len(components)}")

        start_time = time.time()
        sampled, energy, _, _ = self.simulate_quantum_annealing(Q, num_reads)
        sampling_time = time.time() - start_time
        post_start = time.time()
        assignments = self.classical_postprocessing(sampled)
        post_time = time.time() - post_start

        self.quantum_solution = sampled
        self.classical_solution = assignments
        self.performance_metrics = {
            "sampling_time": sampling_time,
            "postprocessing_time": post_time,
            "total_time": sampling_time + post_time,
            "sampled_energy": energy,
            "items_assigned": len(assignments),
            "logical_variables": len(sampled),
            "solution_quality": self.evaluate_solution_quality(assignments),
        }
        print(f"Items assigned: {len(assignments)}/{self.n_items}")
        print(f"Assignment quality score: {self.performance_metrics['solution_quality']:.3f}")
        print("Timing describes this classical demonstration only.")
        return assignments

    def simulate_classical_greedy(self):
        """Place larger-volume items first using a simple grid-order scan."""
        assignments = {}
        item_order = sorted(
            range(self.n_items),
            key=lambda idx: np.prod(self.items[idx]),
            reverse=True,
        )
        for item_idx in item_order:
            pos_idx = self.find_best_feasible_position(
                item_idx, assignments=assignments
            )
            if pos_idx is not None:
                assignments[item_idx] = pos_idx
        return assignments

    def compare_with_classical(self):
        """Compare the postprocessed sample with the included greedy baseline."""
        if self.classical_solution is None:
            raise RuntimeError("Run solve_quantum_packing before comparing results.")
        greedy = self.simulate_classical_greedy()
        print("\nClassical greedy reference:")
        print(f"  Items assigned: {len(greedy)}/{self.n_items}")
        print(f"  Quality score: {self.evaluate_solution_quality(greedy):.3f}")
        print("Postprocessed random-sample result:")
        print(f"  Items assigned: {len(self.classical_solution)}/{self.n_items}")
        print(f"  Quality score: {self.performance_metrics['solution_quality']:.3f}")
        print("This comparison is not a quantum-versus-classical benchmark.")
        return greedy

    def visualize_quantum_solution(self, assignments, title="Discretized 3D Packing"):
        """Plot assigned item boxes and the bin boundary."""
        fig = plt.figure(figsize=(10, 7))
        ax = fig.add_subplot(111, projection="3d")
        self.draw_bin_edges(ax)
        colors = plt.get_cmap("tab10")
        for item_idx, pos_idx in assignments.items():
            lower, upper = self._placement_box(item_idx, pos_idx)
            self.draw_box(
                ax, *lower,
                upper[0] - lower[0], upper[1] - lower[1], upper[2] - lower[2],
                color=colors(item_idx % 10), label=f"Item {item_idx + 1}",
            )
        ax.set_xlabel("Length (X; length units)")
        ax.set_ylabel("Width (Y; length units)")
        ax.set_zlabel("Height (Z; length units)")
        ax.set_title(f"{title}\nAssigned: {len(assignments)}/{self.n_items}")
        ax.set_xlim(0, self.L)
        ax.set_ylim(0, self.W)
        ax.set_zlim(0, self.H)
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend(handles, labels)
        plt.tight_layout()
        plt.show()
        self.print_quantum_solution_details(assignments)

    def draw_bin_edges(self, ax):
        """Draw the twelve edges of the rectangular bin."""
        vertices = [
            (0, 0, 0), (self.L, 0, 0), (self.L, self.W, 0), (0, self.W, 0),
            (0, 0, self.H), (self.L, 0, self.H),
            (self.L, self.W, self.H), (0, self.W, self.H),
        ]
        edges = [
            (0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6),
            (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7),
        ]
        for first, second in edges:
            points = [vertices[first], vertices[second]]
            ax.plot(*zip(*points), color="black", alpha=0.45, linewidth=1)

    def draw_box(self, ax, x, y, z, length, width, height,
                 color="red", alpha=0.65, label=""):
        """Add a translucent axis-aligned box to a 3D plot."""
        vertices = [
            (x, y, z), (x + length, y, z), (x + length, y + width, z),
            (x, y + width, z), (x, y, z + height),
            (x + length, y, z + height),
            (x + length, y + width, z + height),
            (x, y + width, z + height),
        ]
        faces = [
            [vertices[i] for i in face]
            for face in [(0, 1, 5, 4), (2, 3, 7, 6), (0, 3, 7, 4),
                         (1, 2, 6, 5), (4, 5, 6, 7), (0, 1, 2, 3)]
        ]
        collection = Poly3DCollection(
            faces, alpha=alpha, facecolor=color, edgecolor="black", label=label
        )
        ax.add_collection3d(collection)

    def print_quantum_solution_details(self, assignments):
        """Report placements, geometric feasibility, and volume utilization."""
        total_volume = sum(np.prod(item) for item in self.items)
        assigned_volume = sum(np.prod(self.items[i]) for i in assignments)
        bin_volume = self.L * self.W * self.H
        feasible = all(
            not self.check_boundary_violation(item_idx, pos_idx)
            for item_idx, pos_idx in assignments.items()
        ) and all(
            not self.check_overlap(i, assignments[i], j, assignments[j])
            for i, j in combinations(assignments, 2)
        )
        print(f"\nGeometrically feasible assigned placements: {feasible}")
        print(f"Assigned items: {len(assignments)}/{self.n_items}")
        print(f"Assigned-item volume utilization: {assigned_volume / bin_volume:.2%}")
        print(f"All-item volume / bin volume: {total_volume / bin_volume:.2%}")
        for item_idx, pos_idx in sorted(assignments.items()):
            lower, _ = self._placement_box(item_idx, pos_idx)
            print(f"  Item {item_idx + 1}: lower corner {tuple(round(v, 2) for v in lower)}")

    def analyze_quantum_advantage(self):
        """Show logical-variable growth without asserting quantum speedup."""
        item_counts = np.array([1, 5, 10, 20, 50])
        variables = item_counts * self.n_positions
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(item_counts, variables, marker="o")
        ax.set_xlabel("Number of items")
        ax.set_ylabel("Logical binary variables")
        ax.set_title("QUBO variable count for the fixed candidate grid")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        print("Variable count grows as item count × candidate-position count.")
        print("This plot is a model-size analysis, not a hardware feasibility or runtime estimate.")

In [ ]:
items = [
    (3, 2, 2),
    (4, 3, 3),
    (2, 4, 2),
    (5, 2, 3),
    (3, 3, 4),
]
bin_dims = (10, 8, 6)

total_item_volume = sum(np.prod(item) for item in items)
bin_volume = np.prod(bin_dims)
print("3D packing example")
print(f"Items: {len(items)}")
print(f"Bin dimensions: {bin_dims} length units")
print(f"Total item volume: {total_item_volume:.0f} cubic units")
print(f"Bin volume: {bin_volume:.0f} cubic units")
print(f"Candidate grid positions at 1-unit resolution: {np.prod(bin_dims):.0f}")
print("The volume ratio is a necessary capacity check, not a proof of packability.")

In [ ]:
quantum_solver = QuantumQUBOPacking(items, bin_dims, grid_resolution=1.0)
quantum_assignments = quantum_solver.solve_quantum_packing(num_reads=1000)

In [ ]:
quantum_solver.visualize_quantum_solution(
    quantum_assignments,
    title="Postprocessed Classical QUBO-Sampling Demonstration",
)

In [ ]:
greedy_assignments = quantum_solver.compare_with_classical()
print(f"\nGreedy reference placements: {len(greedy_assignments)}")

In [ ]:
quantum_solver.analyze_quantum_advantage()

In [ ]:
Q = quantum_solver.Q_matrix
print("QUBO matrix diagnostics")
print(f"Shape: {Q.shape}")
print(f"Stored entries: {Q.size:,}")
print(f"Non-zero entries: {np.count_nonzero(Q):,}")
print(f"Density: {np.count_nonzero(Q) / Q.size:.2%}")
print(f"Symmetric: {np.allclose(Q, Q.T)}")

diagonal = np.diag(Q)
off_diagonal = Q[~np.eye(Q.shape[0], dtype=bool)]
nonzero_off_diagonal = off_diagonal[off_diagonal != 0]
print(f"Diagonal coefficient range: [{diagonal.min():.2f}, {diagonal.max():.2f}]")
if nonzero_off_diagonal.size:
    print(
        "Nonzero off-diagonal coefficient range: "
        f"[{nonzero_off_diagonal.min():.2f}, {nonzero_off_diagonal.max():.2f}]"
    )

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.spy(Q, markersize=0.5)
ax1.set_title("QUBO Matrix Nonzero Pattern")
ax1.set_xlabel("Variable index")
ax1.set_ylabel("Variable index")
ax2.hist(diagonal, bins=40, alpha=0.7, label="Diagonal", color="steelblue")
if nonzero_off_diagonal.size:
    ax2.hist(
        nonzero_off_diagonal, bins=40, alpha=0.7,
        label="Off-diagonal", color="darkorange",
    )
ax2.set_xlabel("QUBO coefficient")
ax2.set_ylabel("Frequency")
ax2.set_title("Coefficient Distribution")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("QUBO energy landscape: classical random samples")
if quantum_solver.quantum_solution is None:
    print("Run the solver demonstration before inspecting sample energies.")
else:
    Q = quantum_solver.Q_matrix
    sample_count = 1000
    samples = np.random.randint(0, 2, size=(sample_count, Q.shape[0]))
    energies = np.einsum("bi,ij,bj->b", samples, Q, samples, optimize=True)
    sampled_energy = quantum_solver.performance_metrics["sampled_energy"]
    print(f"Random samples: {sample_count}")
    print(f"Energy range: [{energies.min():.2f}, {energies.max():.2f}]")
    print(f"Mean energy: {energies.mean():.2f}")
    print(f"Lowest energy retained by the solver: {sampled_energy:.2f}")
    print("The distribution describes random binary vectors, not annealing dynamics.")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(energies, bins=40, color="steelblue", edgecolor="white", alpha=0.8)
    ax.axvline(sampled_energy, color="darkred", linestyle="--", label="Retained sample energy")
    ax.set_xlabel("QUBO energy")
    ax.set_ylabel("Sample count")
    ax.set_title("Energy Distribution of Classical Random Samples")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Reading the results

The matrix diagnostics describe the coefficients and storage size of the constructed QUBO. The energy histogram describes the sampled binary vectors; a lower energy means a better value for the specified penalty model, but does not by itself prove that a vector corresponds to a feasible packing. The postprocessing stage checks item boundaries and pairwise overlap directly, then reports how many items were placed.

Volume utilization is computed from the volumes of items actually assigned to the bin. It is a volume ratio, not a measure of geometric compactness, stability, or solution optimality. A feasible layout may have empty space, and a high volume ratio does not guarantee that the fixed-orientation items can be arranged feasibly.

The logical variable count is \(n m\), where \(n\) is the item count and \(m\) is the number of candidate grid positions. Dense storage for \(Q\) scales as \((nm)^2\). These counts explain why a fine grid can make this formulation large; they do not establish a particular quantum device's capacity, sampling time, or advantage. Real quantum performance would require an actual hardware workflow, embedding and connectivity considerations, and controlled comparisons against appropriate classical methods. None of those measurements is performed here.

### Further study

Natural model extensions include adding item orientations, multiple bins, load-bearing rules, or accessibility requirements. Each extension would require additional decision variables or constraints and a corresponding analysis of the resulting QUBO size and penalty design. They are not implemented in this demonstration.